In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pandas openpyxl pyarrow xarray netCDF4 h5py tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.4 MB/s eta 0:00:00


In [3]:
import os
import json
import math
import mimetypes
import traceback
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm import tqdm

ROOT = Path("/content/drive/MyDrive/Dataset/PHD")

REPORT_DIR = ROOT / "_dataset_analysis_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset root:", ROOT)
print("Report folder:", REPORT_DIR)

if not ROOT.exists():
    raise FileNotFoundError(f"Dataset path not found: {ROOT}")

Dataset root: /content/drive/MyDrive/Dataset/PHD
Report folder: /content/drive/MyDrive/Dataset/PHD/_dataset_analysis_report


In [4]:
def format_size(bytes_size):
    if bytes_size is None:
        return None
    if bytes_size == 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    i = int(math.floor(math.log(bytes_size, 1024))) if bytes_size > 0 else 0
    i = min(i, len(units) - 1)
    return f"{bytes_size / (1024 ** i):.2f} {units[i]}"


def get_top_folder(file_path, root):
    rel = file_path.relative_to(root)
    parts = rel.parts
    return parts[0] if len(parts) > 1 else "__root__"


def read_file_signature(file_path, n=512):
    try:
        with open(file_path, "rb") as f:
            return f.read(n)
    except Exception:
        return b""


def detect_file_nature(file_path):
    """
    Detect common wrong-download problems:
    - HTML saved as .nc
    - JSON/text error saved as .nc
    - valid NetCDF/HDF5
    - zip
    - parquet
    """
    sig = read_file_signature(file_path, 512)

    if len(sig) == 0:
        return "empty_or_unreadable"

    sig_strip = sig.strip().lower()

    if sig.startswith(b"CDF"):
        return "valid_netcdf3"
    if sig.startswith(b"\x89HDF\r\n\x1a\n"):
        return "valid_netcdf4_hdf5"
    if sig.startswith(b"PK"):
        return "zip_or_xlsx"
    if b"<html" in sig_strip or b"<!doctype html" in sig_strip:
        return "html_saved_as_data"
    if sig_strip.startswith(b"{") or sig_strip.startswith(b"["):
        return "json_or_api_response"
    if sig_strip.startswith(b"<?xml"):
        return "xml_file"
    if file_path.suffix.lower() in [".csv", ".txt"]:
        return "text_table_or_plain_text"
    if file_path.suffix.lower() == ".parquet":
        return "parquet_file"
    if file_path.suffix.lower() in [".tif", ".tiff"]:
        return "geotiff_or_raster"
    return "unknown_binary_or_other"


def safe_stat(file_path):
    try:
        st = file_path.stat()
        return st.st_size, datetime.fromtimestamp(st.st_mtime)
    except Exception:
        return None, None

In [5]:
all_files = []
all_dirs = []

for path in ROOT.rglob("*"):
    if path == REPORT_DIR:
        continue
    if REPORT_DIR in path.parents:
        continue

    if path.is_dir():
        all_dirs.append(path)
    elif path.is_file():
        all_files.append(path)

print("Total folders:", len(all_dirs))
print("Total files:", len(all_files))

inventory_rows = []

for file_path in tqdm(all_files, desc="Scanning files"):
    size_bytes, modified_time = safe_stat(file_path)
    ext = file_path.suffix.lower() if file_path.suffix else "[no_extension]"
    top_folder = get_top_folder(file_path, ROOT)
    rel_path = str(file_path.relative_to(ROOT))
    mime_type, _ = mimetypes.guess_type(str(file_path))
    nature = detect_file_nature(file_path)

    inventory_rows.append({
        "top_folder": top_folder,
        "relative_path": rel_path,
        "file_name": file_path.name,
        "extension": ext,
        "mime_type": mime_type,
        "file_nature": nature,
        "size_bytes": size_bytes,
        "size_readable": format_size(size_bytes),
        "modified_time": modified_time
    })

file_inventory = pd.DataFrame(inventory_rows)

print("Inventory shape:", file_inventory.shape)
file_inventory.head()

Total folders: 31
Total files: 441


Scanning files: 100%|██████████| 441/441 [05:09<00:00,  1.43it/s]

Inventory shape: (441, 9)


,top_folder,relative_path,file_name,extension,mime_type,file_nature,size_bytes,size_readable,modified_time
0,2. noaa_oisst_output,2. noaa_oisst_output/oisst_daily_basin_summary...,oisst_daily_basin_summary_2020_2024.xlsx,.xlsx,application/vnd.openxmlformats-officedocument....,zip_or_xlsx,284271,277.61 KB,2026-05-16 03:29:59
1,2. noaa_oisst_output,2. noaa_oisst_output/oisst_daily_basin_summary...,oisst_daily_basin_summary_2020_2024.csv,.csv,text/csv,text_table_or_plain_text,459602,448.83 KB,2026-05-16 03:29:59
2,2. noaa_oisst_output,2. noaa_oisst_output/oisst_bob_as_2020_2024_co...,oisst_bob_as_2020_2024_combined.parquet,.parquet,None,parquet_file,39391054,37.57 MB,2026-05-16 03:29:59
3,2. noaa_oisst_output,2. noaa_oisst_output/oisst_bob_as_2020_2024_co...,oisst_bob_as_2020_2024_combined.csv,.csv,text/csv,text_table_or_plain_text,1096802620,1.02 GB,2026-05-16 03:29:59
4,2. noaa_oisst_output,2. noaa_oisst_output/noaa_oisst_20260427_globa...,noaa_oisst_20260427_global.xlsx,.xlsx,application/vnd.openxmlformats-officedocument....,zip_or_xlsx,27941800,26.65 MB,2026-05-16 03:29:59


In [6]:
folder_summary = (
    file_inventory
    .groupby("top_folder", as_index=False)
    .agg(
        total_files=("file_name", "count"),
        total_size_bytes=("size_bytes", "sum"),
        file_types=("extension", lambda x: ", ".join(sorted(set(x)))),
        suspicious_files=("file_nature", lambda x: sum(v in ["html_saved_as_data", "empty_or_unreadable", "json_or_api_response"] for v in x)),
        netcdf_files=("extension", lambda x: sum(v in [".nc", ".nc4"] for v in x)),
        csv_files=("extension", lambda x: sum(v == ".csv" for v in x)),
        excel_files=("extension", lambda x: sum(v in [".xlsx", ".xls"] for v in x)),
        parquet_files=("extension", lambda x: sum(v == ".parquet" for v in x)),
        zip_files=("extension", lambda x: sum(v == ".zip" for v in x))
    )
)

folder_summary["total_size_readable"] = folder_summary["total_size_bytes"].apply(format_size)
folder_summary = folder_summary.sort_values("total_size_bytes", ascending=False)

folder_summary

,top_folder,total_files,total_size_bytes,file_types,suspicious_files,netcdf_files,csv_files,excel_files,parquet_files,zip_files,total_size_readable
3,4. Copernicus Marine Global Ocean Waves Reana...,19,6953023043,".csv, .nc, .parquet, .xlsx",0,10,4,2,3,0,6.48 GB
4,5 era5_atmosphere,125,6110207239,".csv, .nc, .parquet, .xlsx",0,120,2,1,2,0,5.69 GB
1,2. noaa_oisst_output,239,2831556684,".csv, .parquet, .xlsx",0,0,119,118,2,0,2.64 GB
0,1. argo_core_profiles_bob_as_2020_2024,18,1502324505,".csv, .nc, .parquet, .xlsx",0,10,3,2,3,0,1.40 GB
2,3. modis_aqua_chlorophyll_2020_2024_processed,5,182325292,".csv, .parquet, .xlsx",0,0,2,1,2,0,173.88 MB
7,8. incois_omni_rama_validation_data,19,51968176,".csv, .parquet, .xlsx",0,0,17,1,1,0,49.56 MB
5,6. ibtracs_cyclone_data_2020_2024,10,28527283,".csv, .parquet, .xlsx",0,0,4,3,3,0,27.21 MB
6,7. gebco_bathymetry_bob_as,6,27962209,".csv, .parquet, .tif, .xlsx",0,0,1,2,1,0,26.67 MB


In [7]:
type_summary = (
    file_inventory
    .groupby(["extension", "file_nature"], as_index=False)
    .agg(
        file_count=("file_name", "count"),
        total_size_bytes=("size_bytes", "sum")
    )
)

type_summary["total_size_readable"] = type_summary["total_size_bytes"].apply(format_size)
type_summary = type_summary.sort_values("total_size_bytes", ascending=False)

type_summary

,extension,file_nature,file_count,total_size_bytes,total_size_readable
0,.csv,text_table_or_plain_text,152,11144776197,10.38 GB
1,.nc,valid_netcdf4_hdf5,140,4532509312,4.22 GB
2,.parquet,parquet_file,17,1433271935,1.33 GB
4,.xlsx,zip_or_xlsx,130,550485248,524.98 MB
3,.tif,geotiff_or_raster,2,26851739,25.61 MB


In [8]:
suspicious_files = file_inventory[
    (file_inventory["file_nature"].isin(["html_saved_as_data", "empty_or_unreadable", "json_or_api_response"])) |
    (file_inventory["size_bytes"].fillna(0) < 1000)
].copy()

print("Suspicious files:", len(suspicious_files))
suspicious_files.head(20)

Suspicious files: 0


,top_folder,relative_path,file_name,extension,mime_type,file_nature,size_bytes,size_readable,modified_time


In [9]:
import xarray as xr

netcdf_rows = []

netcdf_files = [
    ROOT / p for p in file_inventory[
        file_inventory["extension"].isin([".nc", ".nc4"]) &
        file_inventory["file_nature"].isin(["valid_netcdf3", "valid_netcdf4_hdf5"])
    ]["relative_path"].tolist()
]

print("Valid NetCDF files to inspect:", len(netcdf_files))

def get_coord_range(ds, possible_names):
    for name in possible_names:
        if name in ds.coords:
            vals = ds[name].values
            try:
                return float(np.nanmin(vals)), float(np.nanmax(vals))
            except Exception:
                return None, None
        if name in ds.variables:
            vals = ds[name].values
            try:
                return float(np.nanmin(vals)), float(np.nanmax(vals))
            except Exception:
                return None, None
    return None, None


for file_path in tqdm(netcdf_files, desc="Inspecting NetCDF"):
    row = {
        "top_folder": get_top_folder(file_path, ROOT),
        "relative_path": str(file_path.relative_to(ROOT)),
        "file_name": file_path.name,
        "status": "ok",
        "error": None
    }

    try:
        ds = xr.open_dataset(file_path)

        row["data_vars"] = ", ".join(list(ds.data_vars))
        row["coords"] = ", ".join(list(ds.coords))
        row["dims"] = json.dumps({k: int(v) for k, v in ds.sizes.items()})

        lat_min, lat_max = get_coord_range(ds, ["lat", "latitude", "nav_lat", "y"])
        lon_min, lon_max = get_coord_range(ds, ["lon", "longitude", "nav_lon", "x"])
        depth_min, depth_max = get_coord_range(ds, ["depth", "deptht", "lev", "pressure", "pres"])
        time_min, time_max = None, None

        for tname in ["time", "valid_time", "TIME", "JULD"]:
            if tname in ds.coords or tname in ds.variables:
                try:
                    tvals = pd.to_datetime(ds[tname].values)
                    time_min = str(pd.Series(tvals).min())
                    time_max = str(pd.Series(tvals).max())
                    break
                except Exception:
                    pass

        row["lat_min"] = lat_min
        row["lat_max"] = lat_max
        row["lon_min"] = lon_min
        row["lon_max"] = lon_max
        row["depth_min"] = depth_min
        row["depth_max"] = depth_max
        row["time_min"] = time_min
        row["time_max"] = time_max

        ds.close()

    except Exception as e:
        row["status"] = "failed"
        row["error"] = str(e)

    netcdf_rows.append(row)

netcdf_summary = pd.DataFrame(netcdf_rows)

print("NetCDF summary shape:", netcdf_summary.shape)
netcdf_summary.head()

Valid NetCDF files to inspect: 140


Inspecting NetCDF: 100%|██████████| 140/140 [03:26<00:00,  1.48s/it]

NetCDF summary shape: (140, 16)


,top_folder,relative_path,file_name,status,error,data_vars,coords,dims,lat_min,lat_max,lon_min,lon_max,depth_min,depth_max,time_min,time_max
0,1. argo_core_profiles_bob_as_2020_2024,1. argo_core_profiles_bob_as_2020_2024/raw_cor...,ARGO_CORE_as_2022_0_1000dbar.nc,ok,None,"CONFIG_MISSION_NUMBER, CYCLE_NUMBER, DATA_MODE...","LATITUDE, LONGITUDE, TIME, N_POINTS","{""N_POINTS"": 434927}",NaN,NaN,NaN,NaN,None,None,2022-01-01 07:10:27,2022-12-31 15:49:01
1,1. argo_core_profiles_bob_as_2020_2024,1. argo_core_profiles_bob_as_2020_2024/raw_cor...,ARGO_CORE_as_2021_0_1000dbar.nc,ok,None,"CONFIG_MISSION_NUMBER, CYCLE_NUMBER, DATA_MODE...","LATITUDE, LONGITUDE, TIME, N_POINTS","{""N_POINTS"": 613440}",NaN,NaN,NaN,NaN,None,None,2021-01-01 04:08:09,2021-12-31 14:15:30
2,1. argo_core_profiles_bob_as_2020_2024,1. argo_core_profiles_bob_as_2020_2024/raw_cor...,ARGO_CORE_as_2020_0_1000dbar.nc,ok,None,"CONFIG_MISSION_NUMBER, CYCLE_NUMBER, DATA_MODE...","LATITUDE, LONGITUDE, TIME, N_POINTS","{""N_POINTS"": 662291}",NaN,NaN,NaN,NaN,None,None,2020-01-01 00:33:41,2020-12-31 22:34:13
3,1. argo_core_profiles_bob_as_2020_2024,1. argo_core_profiles_bob_as_2020_2024/raw_cor...,ARGO_CORE_bob_2022_0_1000dbar.nc,ok,None,"CONFIG_MISSION_NUMBER, CYCLE_NUMBER, DATA_MODE...","LATITUDE, LONGITUDE, TIME, N_POINTS","{""N_POINTS"": 99240}",NaN,NaN,NaN,NaN,None,None,2022-01-03 04:36:00,2022-12-31 20:56:23
4,1. argo_core_profiles_bob_as_2020_2024,1. argo_core_profiles_bob_as_2020_2024/raw_cor...,ARGO_CORE_as_2023_0_1000dbar.nc,ok,None,"CONFIG_MISSION_NUMBER, CYCLE_NUMBER, DATA_MODE...","LATITUDE, LONGITUDE, TIME, N_POINTS","{""N_POINTS"": 509318}",NaN,NaN,NaN,NaN,None,None,2023-01-01 10:21:39,2023-12-31 03:27:33


In [10]:
tabular_rows = []

tabular_exts = [".csv", ".xlsx", ".xls", ".parquet"]

tabular_files = [
    ROOT / p for p in file_inventory[
        file_inventory["extension"].isin(tabular_exts)
    ]["relative_path"].tolist()
]

print("Tabular files to inspect:", len(tabular_files))


def inspect_csv(file_path):
    df_sample = pd.read_csv(file_path, nrows=5, low_memory=False)
    columns = df_sample.columns.tolist()

    # Try counting rows efficiently
    try:
        with open(file_path, "rb") as f:
            row_count = sum(1 for _ in f) - 1
    except Exception:
        row_count = None

    return columns, row_count, df_sample


def inspect_excel(file_path):
    xls = pd.ExcelFile(file_path)
    sheet_names = xls.sheet_names
    df_sample = pd.read_excel(file_path, sheet_name=sheet_names[0], nrows=5)
    columns = df_sample.columns.tolist()
    return columns, None, df_sample, sheet_names


def inspect_parquet(file_path):
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(file_path)
    columns = pf.schema.names
    row_count = pf.metadata.num_rows if pf.metadata is not None else None
    return columns, row_count


for file_path in tqdm(tabular_files, desc="Inspecting tabular files"):
    row = {
        "top_folder": get_top_folder(file_path, ROOT),
        "relative_path": str(file_path.relative_to(ROOT)),
        "file_name": file_path.name,
        "extension": file_path.suffix.lower(),
        "status": "ok",
        "error": None,
        "columns": None,
        "column_count": None,
        "row_count": None,
        "sheet_names": None,
        "detected_time_columns": None,
        "detected_lat_columns": None,
        "detected_lon_columns": None,
        "detected_depth_columns": None
    }

    try:
        ext = file_path.suffix.lower()

        if ext == ".csv":
            columns, row_count, _ = inspect_csv(file_path)

        elif ext in [".xlsx", ".xls"]:
            columns, row_count, _, sheet_names = inspect_excel(file_path)
            row["sheet_names"] = ", ".join(sheet_names)

        elif ext == ".parquet":
            columns, row_count = inspect_parquet(file_path)

        else:
            columns, row_count = [], None

        columns_lower = [str(c).lower() for c in columns]

        row["columns"] = ", ".join([str(c) for c in columns])
        row["column_count"] = len(columns)
        row["row_count"] = row_count

        row["detected_time_columns"] = ", ".join([c for c in columns if "time" in str(c).lower() or "date" in str(c).lower()])
        row["detected_lat_columns"] = ", ".join([c for c in columns if "lat" in str(c).lower()])
        row["detected_lon_columns"] = ", ".join([c for c in columns if "lon" in str(c).lower()])
        row["detected_depth_columns"] = ", ".join([c for c in columns if "depth" in str(c).lower() or "pres" in str(c).lower()])

    except Exception as e:
        row["status"] = "failed"
        row["error"] = str(e)

    tabular_rows.append(row)

tabular_summary = pd.DataFrame(tabular_rows)

print("Tabular summary shape:", tabular_summary.shape)
tabular_summary.head()

Tabular files to inspect: 299


Inspecting tabular files: 100%|██████████| 299/299 [07:05<00:00,  1.42s/it]

Tabular summary shape: (299, 14)


,top_folder,relative_path,file_name,extension,status,error,columns,column_count,row_count,sheet_names,detected_time_columns,detected_lat_columns,detected_lon_columns,detected_depth_columns
0,2. noaa_oisst_output,2. noaa_oisst_output/oisst_daily_basin_summary...,oisst_daily_basin_summary_2020_2024.xlsx,.xlsx,ok,None,"basin, time_utc, sst_mean, sst_min, sst_max, s...",9,NaN,Sheet1,time_utc,,,
1,2. noaa_oisst_output,2. noaa_oisst_output/oisst_daily_basin_summary...,oisst_daily_basin_summary_2020_2024.csv,.csv,ok,None,"basin, time_utc, sst_mean, sst_min, sst_max, s...",9,3534.0,None,time_utc,,,
2,2. noaa_oisst_output,2. noaa_oisst_output/oisst_bob_as_2020_2024_co...,oisst_bob_as_2020_2024_combined.parquet,.parquet,ok,None,"time_utc, depth_m, latitude_degrees_north, lon...",8,15860592.0,None,time_utc,latitude_degrees_north,longitude_degrees_east,depth_m
3,2. noaa_oisst_output,2. noaa_oisst_output/oisst_bob_as_2020_2024_co...,oisst_bob_as_2020_2024_combined.csv,.csv,ok,None,"time_utc, depth_m, latitude_degrees_north, lon...",8,15860592.0,None,time_utc,latitude_degrees_north,longitude_degrees_east,depth_m
4,2. noaa_oisst_output,2. noaa_oisst_output/noaa_oisst_20260427_globa...,noaa_oisst_20260427_global.xlsx,.xlsx,ok,None,"time_utc, depth_m, latitude_degrees_north, lon...",8,NaN,Sheet1,time_utc,latitude_degrees_north,longitude_degrees_east,depth_m


In [11]:
expected_keywords = {
    "Argo_Core_Profile": ["argo", "core"],
    "BGC_Argo": ["bgc", "sprof"],
    "NOAA_OISST": ["oisst"],
    "MODIS_Chlorophyll": ["modis", "chlor"],
    "Copernicus_Physics": ["copernicus", "physics"],
    "Copernicus_Waves": ["wave"],
    "ERA5_Atmosphere": ["era5"],
    "IBTrACS_Cyclone": ["ibtracs"],
    "GEBCO_Bathymetry": ["gebco"],
    "INCOIS_OMNI_RAMA": ["rama", "incois", "omni"]
}

readiness_rows = []

all_paths_text = " ".join(file_inventory["relative_path"].astype(str).str.lower().tolist())

for dataset_name, keywords in expected_keywords.items():
    found = any(k in all_paths_text for k in keywords)

    matching_files = file_inventory[
        file_inventory["relative_path"].astype(str).str.lower().apply(
            lambda x: any(k in x for k in keywords)
        )
    ]

    readiness_rows.append({
        "dataset_component": dataset_name,
        "detected": found,
        "matching_file_count": len(matching_files),
        "total_size_bytes": matching_files["size_bytes"].sum() if len(matching_files) > 0 else 0,
        "total_size_readable": format_size(matching_files["size_bytes"].sum()) if len(matching_files) > 0 else "0 B",
        "main_extensions": ", ".join(sorted(set(matching_files["extension"].tolist()))) if len(matching_files) > 0 else ""
    })

readiness_summary = pd.DataFrame(readiness_rows)

readiness_summary

,dataset_component,detected,matching_file_count,total_size_bytes,total_size_readable,main_extensions
0,Argo_Core_Profile,True,18,1502324505,1.40 GB,".csv, .nc, .parquet, .xlsx"
1,BGC_Argo,False,0,0,0 B,
2,NOAA_OISST,True,239,2831556684,2.64 GB,".csv, .parquet, .xlsx"
3,MODIS_Chlorophyll,True,5,182325292,173.88 MB,".csv, .parquet, .xlsx"
4,Copernicus_Physics,True,19,6953023043,6.48 GB,".csv, .nc, .parquet, .xlsx"
5,Copernicus_Waves,True,19,6953023043,6.48 GB,".csv, .nc, .parquet, .xlsx"
6,ERA5_Atmosphere,True,125,6110207239,5.69 GB,".csv, .nc, .parquet, .xlsx"
7,IBTrACS_Cyclone,True,10,28527283,27.21 MB,".csv, .parquet, .xlsx"
8,GEBCO_Bathymetry,True,6,27962209,26.67 MB,".csv, .parquet, .tif, .xlsx"
9,INCOIS_OMNI_RAMA,True,19,51968176,49.56 MB,".csv, .parquet, .xlsx"


In [12]:
report_xlsx = REPORT_DIR / "PHD_dataset_inventory_report.xlsx"

with pd.ExcelWriter(report_xlsx, engine="openpyxl") as writer:
    folder_summary.to_excel(writer, sheet_name="folder_summary", index=False)
    type_summary.to_excel(writer, sheet_name="file_type_summary", index=False)
    file_inventory.to_excel(writer, sheet_name="file_inventory", index=False)
    suspicious_files.to_excel(writer, sheet_name="suspicious_files", index=False)
    netcdf_summary.to_excel(writer, sheet_name="netcdf_summary", index=False)
    tabular_summary.to_excel(writer, sheet_name="tabular_summary", index=False)
    readiness_summary.to_excel(writer, sheet_name="readiness_summary", index=False)

print("Excel report saved:", report_xlsx)

Excel report saved: /content/drive/MyDrive/Dataset/PHD/_dataset_analysis_report/PHD_dataset_inventory_report.xlsx


In [13]:
report_md = REPORT_DIR / "PHD_dataset_inventory_report.md"

total_size = file_inventory["size_bytes"].sum()

md = []
md.append("# PhD Ocean Dataset Inventory Report\n")
md.append(f"Dataset root: `{ROOT}`\n")
md.append(f"Generated on: `{datetime.now()}`\n")
md.append(f"Total folders: **{len(all_dirs)}**\n")
md.append(f"Total files: **{len(all_files)}**\n")
md.append(f"Total size: **{format_size(total_size)}**\n")

md.append("\n## Folder Summary\n")
md.append(folder_summary.to_markdown(index=False))

md.append("\n\n## File Type Summary\n")
md.append(type_summary.to_markdown(index=False))

md.append("\n\n## Dataset Readiness Summary\n")
md.append(readiness_summary.to_markdown(index=False))

md.append("\n\n## Suspicious Files\n")
if len(suspicious_files) > 0:
    md.append(suspicious_files[["top_folder", "relative_path", "file_nature", "size_readable"]].head(50).to_markdown(index=False))
else:
    md.append("No suspicious files detected.")

md.append("\n\n## NetCDF Summary Preview\n")
if len(netcdf_summary) > 0:
    md.append(netcdf_summary.head(30).to_markdown(index=False))
else:
    md.append("No valid NetCDF files detected.")

md.append("\n\n## Tabular Summary Preview\n")
if len(tabular_summary) > 0:
    md.append(tabular_summary.head(30).to_markdown(index=False))
else:
    md.append("No tabular files detected.")

report_md.write_text("\n".join(md), encoding="utf-8")

print("Markdown report saved:", report_md)

Markdown report saved: /content/drive/MyDrive/Dataset/PHD/_dataset_analysis_report/PHD_dataset_inventory_report.md


In [14]:
import shutil
from google.colab import files

zip_name = str(REPORT_DIR / "PHD_dataset_analysis_report")

shutil.make_archive(zip_name, "zip", REPORT_DIR)

files.download(zip_name + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>